In [1]:
%reload_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')
from IPython.display import clear_output

import os, sys
import numpy as np
import pandas as pd
import swifter
import time
from tqdm.auto import tqdm
from rdkit.Chem import BRICS, Descriptors

from rdkit import Chem
from rdkit.Chem import PandasTools
PandasTools.RenderImagesInAllDataFrames(images=True)
from rdkit import RDLogger; RDLogger.DisableLog('rdApp.*')

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates['geass_font'] = go.layout.Template(layout=dict(font=dict(family='DejaVu Sans')))
pio.templates.default = 'plotly_white+geass_font'
pio.renderers.default = "notebook"
pio.renderers.default_config = {
    "toImageButtonOptions": {"format": "png", "scale": 5}}

from rdkit.Chem.Draw import IPythonConsole
IPythonConsole.ipython_useSVG = True
IPythonConsole.molSize = 600, 400
IPythonConsole.drawOptions.addAtomIndices = False

sys.path.append(os.path.abspath(".."))
from new_geass import Geass

clear_output()

In [2]:
from paper_style import apply_paper_style, save_paper_style

palette_1 = ["#0D4A70", "#3082BD", "#9EC9E2", "#E4F1F7"]
palette_2 = ["#0D4A70", "#6189A2", "#A9475B", "#BD7281", "#CF9B26", "#DAB359"]
geodataviz_diverging_1 = ["#045275", "#089099", "#39B185", "#7CCBA2", "#FCDE9C",
                          "#EEB479", "#E88471", "#7C1D6F", "#CF597E", "#9A275A"]
node_colors = ['#d10808', '#ffc32d','#009b73']

In [ ]:
df_NPA = pd.read_csv('../data/NPAtlas_download_2024_09.tsv', sep='\t')
df_NPA = df_NPA[~df_NPA['compound_smiles'].str.contains(R'\.')].reset_index(drop=True)
df_NPA = df_NPA[['npaid', 'compound_smiles']].rename(
    columns={'npaid': 'ID', 'compound_smiles': 'ISOSMILES'})
PandasTools.AddMoleculeColumnToFrame(df_NPA, 'ISOSMILES')

BENCH_SEED = 42
WALL_SAMPLE_SIZE = 10200
WALL_REPEATS = 5
METHOD_SAMPLE_SIZE = 300
METHOD_REPEATS = 5

def _sample_profile(df):
    mols = df['ROMol'].tolist()
    heavy_atoms = np.array([mol.GetNumHeavyAtoms() for mol in mols], dtype=float)
    mw = np.array([Descriptors.MolWt(mol) for mol in mols], dtype=float)
    return pd.DataFrame({
        'n_molecules': [len(mols)],
        'heavy_atoms_mean': [float(np.mean(heavy_atoms)) if len(heavy_atoms) else np.nan],
        'heavy_atoms_median': [float(np.median(heavy_atoms)) if len(heavy_atoms) else np.nan],
        'heavy_atoms_q1': [float(np.quantile(heavy_atoms, 0.25)) if len(heavy_atoms) else np.nan],
        'heavy_atoms_q3': [float(np.quantile(heavy_atoms, 0.75)) if len(heavy_atoms) else np.nan],
        'mw_mean': [float(np.mean(mw)) if len(mw) else np.nan],
        'mw_median': [float(np.median(mw)) if len(mw) else np.nan],
        'mw_q1': [float(np.quantile(mw, 0.25)) if len(mw) else np.nan],
        'mw_q3': [float(np.quantile(mw, 0.75)) if len(mw) else np.nan],
    })


## Method comparsion

In [ ]:
# Cell A: Build a molecule-level benchmark table (same molecules for all methods)
import metadisassembler

SEED = BENCH_SEED
N = METHOD_SAMPLE_SIZE
N_REPEATS = METHOD_REPEATS

df_eval = df_NPA.sample(N, random_state=SEED).reset_index(drop=True)[["ID", "ISOSMILES", "ROMol"]].copy()
method_profile = _sample_profile(df_eval)

def _run(method, query):
    setup_t0 = time.perf_counter()
    obj = method["setup"]()
    setup_s = time.perf_counter() - setup_t0
    t0 = time.perf_counter()
    try:
        frags = method["run"](obj, query)
        ok, err = True, ""
        if not frags:
            ok, err = False, "empty"
    except Exception as e:
        frags, ok, err = [], False, str(e)
    return frags, setup_s, time.perf_counter() - t0, ok, err

def _geass_run(geass, mol):
    return geass.get_fragments(mol)

def _brics_run(_, mol):
    return list(BRICS.BRICSDecompose(mol, returnMols=True))

def _metadi_run(medi, smi):
    medi.input_query(smi)
    inchis = medi.disassemble(return_inchis=True, write_output=False) or []
    return [mol for x in inchis if isinstance(x, str) and x.strip() and (mol := Chem.MolFromInchi(x)) is not None]

methods = {
    "GEASS": {"setup": lambda: Geass(annotate=False), "run": _geass_run},
    "BRICS": {"setup": lambda: None, "run": _brics_run},
    "MetaDi": {"setup": metadisassembler.MetaDisassembler, "run": _metadi_run},
}

rows = []
for rep in range(N_REPEATS):
    for _, r in tqdm(df_eval.iterrows(), total=len(df_eval), desc=f'Replicate {rep + 1}/{N_REPEATS}'):
        for method, runner in methods.items():
            query = r["ROMol"] if method != "MetaDi" else r["ISOSMILES"]
            frags, setup_s, runtime_s, ok, err = _run(runner, query)

            n_frags = len(frags)
            atom_counts = [m.GetNumAtoms() for m in frags if m is not None]
            mean_atoms = float(np.mean(atom_counts)) if atom_counts else np.nan

            rows.append({
                "replicate": rep + 1,
                "ID": r["ID"],
                "method": method,
                "setup_s": setup_s,
                "runtime_s": runtime_s,
                "total_s": setup_s + runtime_s,
                "ok": ok,
                "error": err,
                "n_frags": n_frags,
                "mean_atoms_per_frag": mean_atoms,
                "heavy_atoms": r["ROMol"].GetNumHeavyAtoms(),
                "mol_wt": Descriptors.MolWt(r["ROMol"]),
            })

bench = pd.DataFrame(rows)
bench.to_csv('fragmentation_benchmark_runs.csv', index=False)





In [23]:
bench = pd.read_csv('fragmentation_benchmark.csv')

bench_summary = bench.groupby('method', as_index=False).agg(
    n_rows=('ID', 'size'),
    mean_n_frags=('n_frags', 'mean'),
    std_n_frags=('n_frags', 'std'),
    median_n_frags=('n_frags', 'median'),
    q1_n_frags=('n_frags', lambda s: s.quantile(0.25)),
    q3_n_frags=('n_frags', lambda s: s.quantile(0.75)),
    mean_atoms_per_frag=('mean_atoms_per_frag', 'mean'),
    std_atoms_per_frag=('mean_atoms_per_frag', 'std'),
    median_atoms_per_frag=('mean_atoms_per_frag', 'median'),
    q1_atoms_per_frag=('mean_atoms_per_frag', lambda s: s.quantile(0.25)),
    q3_atoms_per_frag=('mean_atoms_per_frag', lambda s: s.quantile(0.75)),
)


print(bench_summary)
print(bench.groupby('method')['runtime_s']
    .agg(['median', 'std']))

   method  n_rows  mean_n_frags  std_n_frags  median_n_frags  q1_n_frags  \
0   BRICS     300      4.266667     2.820533             4.0         2.0   
1   GEASS     300      2.960000     3.158849             2.0         1.0   
2  MetaDi     300      3.770000     3.442373             5.0         0.0   

   q3_n_frags  mean_atoms_per_frag  std_atoms_per_frag  median_atoms_per_frag  \
0         6.0            11.095788            5.752854                 9.3875   
1         3.0            16.869618            9.548841                14.0000   
2         7.0             4.127981            1.458983                 3.8750   

   q1_atoms_per_frag  q3_atoms_per_frag  
0            7.09375          13.062500  
1            9.80000          20.000000  
2            3.20000           4.666667  
          median        std
method                     
BRICS   0.002905   0.188390
GEASS   0.069113   0.041184
MetaDi  1.854966  25.585518


In [ ]:
colors = {"GEASS": palette_2[0], "BRICS": palette_2[2], "MetaDi": palette_2[4]}

fig_runtime = go.Figure()
for m in colors:
    x = np.sort(bench.loc[bench["method"] == m, "runtime_s"].values)
    y = np.arange(1, len(x) + 1) / len(x)
    fig_runtime.add_trace(go.Scatter(x=x, y=y, mode="lines", name=m, line=dict(color=colors[m], width=2)))

fig_runtime.update_xaxes(
    type="log", title_text="Runtime per molecule (s, log)",
    automargin=True, tickmode="array", range=[-3.5, 3],
    tickvals=[1e-4, 1e-3, 1e-2, 1e-1, 1e0, 1e1, 1e2, 1e3, 1e4],)
fig_runtime.update_yaxes(title_text="ECDF", automargin=True)

fig_runtime.update_layout(showlegend=True)
fig_runtime.update_layout(legend=dict(x=0.98, y=0.02, xanchor='right', yanchor='bottom',
    bgcolor='rgba(255,255,255,0.85)', bordercolor='black', borderwidth=1))
save_paper_style(fig_runtime, 'figures/fig6a.svg', zoom=1.4)

          median        std
method                     
BRICS   0.002905   0.188390
GEASS   0.069113   0.041184
MetaDi  1.854966  25.585518


In [15]:
fig_dist = go.Figure()
colors = {"GEASS": [palette_2[0], palette_2[1]], "BRICS": [palette_2[2], palette_2[3]], "MetaDi": [palette_2[4], palette_2[5]]}
metrics = [("n_frags", "n. frags"), ("mean_atoms_per_frag", "atoms/frag")]
for method in ["GEASS", "BRICS", "MetaDi"]:
    for j, (col, label) in enumerate(metrics):
        vals = bench.loc[bench["method"] == method, col].dropna()
        color = colors[method][j]
        fig_dist.add_trace(go.Violin(
            x=[f"{method}<br>{label}"] * len(vals), y=vals,
            box_visible=True, meanline_visible=True,
            line_color=color, fillcolor=color,
            showlegend=False,
        ))

fig_dist.update_xaxes(tickmode="array", categoryorder='array', categoryarray=['GEASS<br>n. frags', 'GEASS<br>atoms/frag', 'BRICS<br>n. frags', 'BRICS<br>atoms/frag', 'MetaDi<br>n. frags', 'MetaDi<br>atoms/frag'], tickangle=0, automargin=True)
fig_dist.update_yaxes(title_text="Value")
save_paper_style(fig_dist, 'figures/fig6b.svg', zoom=1.4)

In [16]:
def combine_figs(fig1, fig2):
    from plotly.subplots import make_subplots as _make_subplots

    dic = {'anchor', 'domain', 'overlaying', 'position', 'matches', 'scaleanchor', 'scaleratio', 'constrain', 'constraintoward'}
    w1, h1 = int(fig1.layout.width or 750), int(fig1.layout.height or 470)
    w2, h2 = int(fig2.layout.width or 750), int(fig2.layout.height or 470)
    fig = _make_subplots(rows=1, cols=2, column_widths=[w1, w2], horizontal_spacing=0.08)

    for src, col in ((fig1, 1), (fig2, 2)):
        for tr in src.data:
            fig.add_trace(tr, row=1, col=col)
        for shp in src.layout.shapes or []:
            if getattr(shp, 'xref', None) == 'paper' and getattr(shp, 'yref', None) == 'paper':
                continue
            fig.add_shape(shp, row=1, col=col)
        for ann in src.layout.annotations or []:
            fig.add_annotation(ann, row=1, col=col)

    fig.update_layout(width=w1 + w2 + 70, height=max(h1, h2), showlegend=any(tr.showlegend is not False for src in (fig1, fig2) for tr in src.data))
    fig.update_xaxes(**{k: v for k, v in fig1.layout.xaxis.to_plotly_json().items() if v is not None and k not in dic}, row=1, col=1)
    fig.update_yaxes(**{k: v for k, v in fig1.layout.yaxis.to_plotly_json().items() if v is not None and k not in dic}, row=1, col=1)
    fig.update_xaxes(**{k: v for k, v in fig2.layout.xaxis.to_plotly_json().items() if v is not None and k not in dic}, row=1, col=2)
    fig.update_yaxes(**{k: v for k, v in fig2.layout.yaxis.to_plotly_json().items() if v is not None and k not in dic}, row=1, col=2)
    for ann in fig.layout.annotations[:2]:
        ann.y += 0.04
    return fig

fig = combine_figs(fig_runtime, fig_dist)
save_paper_style(fig, 'figures/fig6.svg', rows=1, cols=2, zoom=1.4)
